
## Folder 05 — File 01: Data Quality Framework:


Instead of writing separate validation logic for every table, this framework uses reusable functions for:

- Record count validation.
- Null and blank checks.
- Duplicate business-key detection.
- Required-column validation.
- Negative-value checks.
- Referential and business-rule checks in later files.
- Quality status and pipeline failure thresholds.


**Purpose**: Build a reusable data-quality framework that checks all seven Silver datasets and produces a centralized quality report.


## Data Quality Architecture:


```text
┌──────────────────────────────────────────────┐
│             SILVER DELTA TABLES              │
│                                              │
│  1. Customers                                │
│  2. Products                                 │
│  3. Orders                                   │
│  4. Order Items                              │
│  5. Payments                                 │
│  6. Shipments                                │
│  7. Inventory Snapshots                      │
└──────────────────────────────────────────────┘
                       │
                       ▼
┌──────────────────────────────────────────────┐
│       REUSABLE DATA QUALITY FRAMEWORK        │
│                                              │
│  • Completeness                              │
│  • Uniqueness                                │
│  • Validity                                  │
│  • Consistency                               │
└──────────────────────────────────────────────┘
                       │
                       ▼
┌──────────────────────────────────────────────┐
│          CENTRALIZED QUALITY REPORT         │
│                                              │
│  • PASS / FAIL Status                        │
│  • Quality Metrics                           │
│  • Failure Reasons                            │
│  • Dataset-wise Quality Results              │
└──────────────────────────────────────────────┘
```

In [0]:
## 1. Import libraries

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
    DoubleType,
    TimestampType
)

from datetime import datetime

print("Libraries imported successfully.")


## 2. Project configuration

CATALOG = "workspace"
SILVER_SCHEMA = "ecommerce_silver"

QUALITY_REPORT_TABLE = (
    f"{CATALOG}.{SILVER_SCHEMA}.data_quality_report"
)

FAIL_PIPELINE_ON_FAILURE = True


## 3. Define dataset quality configurations

QUALITY_CONFIG = {

    "silver_customers": {
        "required_columns": [
            "customer_id",
            "first_name",
            "last_name",
            "email",
            "created_at"
        ],
        "key_columns": [
            "customer_id"
        ],
        "custom_rules": []
    },

    "silver_products": {
        "required_columns": [
            "product_id",
            "product_name",
            "category",
            "unit_price",
            "unit_cost",
            "created_at"
        ],
        "key_columns": [
            "product_id"
        ],
        "custom_rules": [
            (
                "negative_unit_price",
                "unit_price < 0"
            ),
            (
                "negative_unit_cost",
                "unit_cost < 0"
            )
        ]
    },

    "silver_orders": {
        "required_columns": [
            "order_id",
            "customer_id",
            "order_timestamp",
            "order_status",
            "order_total"
        ],
        "key_columns": [
            "order_id"
        ],
        "custom_rules": [
            (
                "negative_order_total",
                "order_total < 0"
            )
        ]
    },

    "silver_order_items": {
        "required_columns": [
            "order_item_id",
            "order_id",
            "product_id",
            "quantity",
            "unit_price",
            "discount_rate",
            "line_total"
        ],
        "key_columns": [
            "order_item_id"
        ],
        "custom_rules": [
            (
                "invalid_quantity",
                "quantity <= 0"
            ),
            (
                "negative_unit_price",
                "unit_price < 0"
            ),
            (
                "invalid_discount_rate",
                "discount_rate < 0 OR discount_rate > 1"
            ),
            (
                "negative_line_total",
                "line_total < 0"
            )
        ]
    },

    "silver_payments": {
        "required_columns": [
            "payment_id",
            "order_id",
            "payment_method",
            "payment_status",
            "amount",
            "payment_timestamp"
        ],
        "key_columns": [
            "payment_id"
        ],
        "custom_rules": [
            (
                "negative_payment_amount",
                "amount < 0"
            )
        ]
    },

    "silver_shipments": {
        "required_columns": [
            "shipment_id",
            "order_id",
            "carrier",
            "shipment_status",
            "shipped_date"
        ],
        "key_columns": [
            "shipment_id"
        ],
        "custom_rules": [
            (
                "delivery_before_shipment",
                "delivered_date IS NOT NULL "
                "AND delivered_date < shipped_date"
            ),
            (
                "delivered_without_delivery_date",
                "shipment_status = 'DELIVERED' "
                "AND delivered_date IS NULL"
            )
        ]
    },

    "silver_inventory": {
        "required_columns": [
            "inventory_snapshot_id",
            "product_id",
            "warehouse_id",
            "quantity_on_hand",
            "snapshot_date"
        ],
        "key_columns": [
            "product_id",
            "warehouse_id",
            "snapshot_date"
        ],
        "custom_rules": [
            (
                "negative_inventory_quantity",
                "quantity_on_hand < 0"
            )
        ]
    }
}

print(f"Configured datasets: {len(QUALITY_CONFIG)}")


## 4. Create Silver schema

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS "
    f"`{CATALOG}`.`{SILVER_SCHEMA}`"
)


## 5. Define reusable quality check function

def run_quality_check(
    dataset_name,
    check_name,
    failed_records,
    total_records,
    details
):

    failure_percentage = (
        round(
            failed_records / total_records * 100,
            2
        )
        if total_records > 0
        else 0.0
    )

    # All configured checks have a strict zero-failure threshold.

    status = (
        "PASS"
        if failed_records == 0 and total_records > 0
        else "FAIL"
    )

    return {
        "dataset_name": dataset_name,
        "check_name": check_name,
        "total_records": int(total_records),
        "failed_records": int(failed_records),
        "failure_percentage": float(failure_percentage),
        "status": status,
        "details": details,
        "checked_at": datetime.now()
    }


## 6. Define function to validate one dataset

def validate_dataset(
    dataset_name,
    config
):

    table_name = (
        f"{CATALOG}.{SILVER_SCHEMA}.{dataset_name}"
    )

    if not spark.catalog.tableExists(table_name):
        raise Exception(
            f"Silver table not found: {table_name}"
        )

    df = spark.table(table_name)

    total_records = df.count()

    results = []

    available_columns = set(df.columns)

    # --------------------------------------------------
    # Check 1: Table is not empty
    # --------------------------------------------------

    results.append(
        run_quality_check(
            dataset_name,
            "non_empty_table",
            0 if total_records > 0 else 1,
            total_records,
            "Table must contain at least one record."
        )
    )

    # --------------------------------------------------
    # Check 2: Required columns exist
    # --------------------------------------------------

    required_columns = config["required_columns"]

    missing_columns = [
        column
        for column in required_columns
        if column not in available_columns
    ]

    results.append(
        run_quality_check(
            dataset_name,
            "required_columns_exist",
            len(missing_columns),
            len(required_columns),
            (
                "Missing columns: "
                + str(missing_columns)
                if missing_columns
                else "All required columns exist."
            )
        )
    )

    # Do not execute column-level checks when

    # the table schema is incomplete.

    if missing_columns:
        return results

    # --------------------------------------------------
    # Check 3: Required values are not null or blank
    # --------------------------------------------------

    required_condition = None

    for column in required_columns:

        column_condition = (
            F.col(column).isNull()
            |
            (
                F.trim(
                    F.col(column).cast("string")
                ) == ""
            )
        )

        required_condition = (
            column_condition
            if required_condition is None
            else required_condition | column_condition
        )

    incomplete_records = (
        df
        .filter(required_condition)
        .count()
    )

    results.append(
        run_quality_check(
            dataset_name,
            "required_values_completeness",
            incomplete_records,
            total_records,
            "Records with at least one missing or blank required value."
        )
    )

    # --------------------------------------------------
    # Check 4: Duplicate business keys
    # --------------------------------------------------

    key_columns = config["key_columns"]

    duplicate_groups = (
        df
        .groupBy(*key_columns)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    results.append(
        run_quality_check(
            dataset_name,
            "duplicate_business_keys",
            duplicate_groups,
            total_records,
            (
                "Duplicate key groups for: "
                + ", ".join(key_columns)
            )
        )
    )

    # --------------------------------------------------
    # Check 5: Custom business rules
    # --------------------------------------------------

    for rule_name, rule_expression in config["custom_rules"]:

        # Ensure all referenced columns exist.

        # The expressions in this configuration are

        # controlled by this notebook.

        failed_count = (
            df
            .filter(F.expr(rule_expression))
            .count()
        )

        results.append(
            run_quality_check(
                dataset_name,
                rule_name,
                failed_count,
                total_records,
                f"Invalid condition: {rule_expression}"
            )
        )

    return results


 ## 7. Run quality checks for all seven datasets

all_quality_results = []

for dataset_name, config in QUALITY_CONFIG.items():

    print("=" * 60)
    print(f"Validating: {dataset_name}")
    print("=" * 60)

    dataset_results = validate_dataset(
        dataset_name,
        config
    )

    all_quality_results.extend(dataset_results)

    for result in dataset_results:

        print(
            f"{result['check_name']}: "
            f"{result['status']} "
            f"(Failed: {result['failed_records']})"
        )


## 8. Create centralized quality report

quality_schema = StructType([

    StructField(
        "dataset_name",
        StringType(),
        False
    ),

    StructField(
        "check_name",
        StringType(),
        False
    ),

    StructField(
        "total_records",
        LongType(),
        False
    ),

    StructField(
        "failed_records",
        LongType(),
        False
    ),

    StructField(
        "failure_percentage",
        DoubleType(),
        False
    ),

    StructField(
        "status",
        StringType(),
        False
    ),

    StructField(
        "details",
        StringType(),
        True
    ),

    StructField(
        "checked_at",
        TimestampType(),
        False
    )
])

quality_report_df = spark.createDataFrame(
    all_quality_results,
    schema=quality_schema
)

display(quality_report_df)


## 9. Save quality report as Delta table

(
    quality_report_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(QUALITY_REPORT_TABLE)
)

print(f"Quality report saved: {QUALITY_REPORT_TABLE}")


## 10. Dataset-level quality summary

dataset_summary_df = (
    quality_report_df
    .groupBy("dataset_name")
    .agg(
        F.count("*").alias("total_checks"),

        F.sum(
            F.when(
                F.col("status") == "PASS",
                1
            ).otherwise(0)
        ).alias("passed_checks"),

        F.sum(
            F.when(
                F.col("status") == "FAIL",
                1
            ).otherwise(0)
        ).alias("failed_checks")
    )
    .withColumn(
        "overall_status",
        F.when(
            F.col("failed_checks") == 0,
            "PASS"
        ).otherwise("FAIL")
    )
    .orderBy("dataset_name")
)

display(dataset_summary_df)


## 11. Overall quality metrics

total_checks = quality_report_df.count()

passed_checks = (
    quality_report_df
    .filter(F.col("status") == "PASS")
    .count()
)

failed_checks = (
    quality_report_df
    .filter(F.col("status") == "FAIL")
    .count()
)

quality_score = (
    round(
        passed_checks / total_checks * 100,
        2
    )
    if total_checks > 0
    else 0.0
)

print("=" * 60)
print("OVERALL DATA QUALITY SUMMARY")
print("=" * 60)

print(f"Total checks  : {total_checks}")
print(f"Passed checks : {passed_checks}")
print(f"Failed checks : {failed_checks}")
print(f"Quality score : {quality_score}%")

print("=" * 60)


## 12. Fail pipeline when critical checks fail

if FAIL_PIPELINE_ON_FAILURE and failed_checks > 0:

    raise Exception(
        f"Data quality validation failed. "
        f"{failed_checks} of {total_checks} checks failed. "
        f"Review {QUALITY_REPORT_TABLE}."
    )

print("All configured data quality checks passed.")


## 13. Final output

display(
    spark.table(QUALITY_REPORT_TABLE)
    .orderBy(
        "dataset_name",
        "check_name"
    )
)

print("=" * 60)
print("DATA QUALITY FRAMEWORK COMPLETED")
print("=" * 60)
print(f"Quality report: {QUALITY_REPORT_TABLE}")
print(f"Quality score : {quality_score}%")
print("=" * 60)


In [0]:
# Display only failed quality checks

failed_quality_df = (
    spark.table("workspace.ecommerce_silver.data_quality_report")
    .filter(F.col("status") == "FAIL")
    .orderBy("dataset_name", "check_name")
)

display(failed_quality_df)

In [0]:
# Identify records with missing required values

missing_condition = None

for column in required_columns:

    condition = (
        F.col(column).isNull() |
        (
            F.trim(F.col(column).cast("string")) == ""
        )
    )

    missing_condition = (
        condition
        if missing_condition is None
        else missing_condition | condition
    )

invalid_customers = (
    customer_df
    .filter(missing_condition)
)

print("Invalid customer records:", invalid_customers.count())

display(invalid_customers)


## Execute the correction:

In [0]:
from pyspark.sql import functions as F

CATALOG = "workspace"

SILVER_TABLE = (
    "workspace.ecommerce_silver.silver_customers"
)

QUARANTINE_TABLE = (
    "workspace.ecommerce_silver.quarantine_customers"
)

# Read current Silver customers
customer_df = spark.table(SILVER_TABLE)

# Identify invalid records
invalid_condition = (
    F.col("email").isNull() |
    (F.trim(F.col("email")) == "")
)

# Separate valid and invalid records
valid_customers = customer_df.filter(
    ~invalid_condition
)

quarantined_customers = (
    customer_df
    .filter(invalid_condition)
    .withColumn(
        "quarantine_reason",
        F.lit("Missing or blank email")
    )
    .withColumn(
        "quarantined_at",
        F.current_timestamp()
    )
)

# Verify counts before writing
print("Original records:", customer_df.count())
print("Valid records:", valid_customers.count())
print("Quarantined records:", quarantined_customers.count())

# Save valid customers back to Silver
(
    valid_customers.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLE)
)

# Save quarantined records separately
(
    quarantined_customers.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(QUARANTINE_TABLE)
)

print("Customer quarantine completed.")

In [0]:
customer_df = spark.table(
    "workspace.ecommerce_silver.silver_customers"
)

# Important: the six invalid records were removed from Silver.
# Recover them from the original Bronze dataset.

bronze_customers = (
    spark.read
    .format("delta")
    .load(
        "/Volumes/workspace/default/data_volume/"
        "ecommerce_lakehouse/bronze/customers"
    )
)

invalid_condition = (
    F.col("email").isNull() |
    (F.trim(F.col("email")) == "")
)

quarantined_customers = (
    bronze_customers
    .filter(invalid_condition)
    .withColumn(
        "quarantine_reason",
        F.lit("Missing or blank email")
    )
    .withColumn(
        "quarantined_at",
        F.current_timestamp()
    )
)

print("Recovered invalid records:", quarantined_customers.count())

(
    quarantined_customers.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(QUARANTINE_TABLE)
)

display(spark.table(QUARANTINE_TABLE))


## Verify the results:

In [0]:
tables = {
    "Bronze Customers": (
        "/Volumes/workspace/default/data_volume/"
        "ecommerce_lakehouse/bronze/customers"
    ),
    "Silver Customers": (
        "workspace.ecommerce_silver.silver_customers"
    ),
    "Quarantine Customers": (
        "workspace.ecommerce_silver.quarantine_customers"
    )
}

for name, source in tables.items():

    if name == "Bronze Customers":
        df = spark.read.format("delta").load(source)
    else:
        df = spark.table(source)

    print(f"{name}: {df.count()}")